In [90]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time
from json import load as load_json
from bs4.element import Tag
from typing import Optional, List

In [27]:
_Delay = 2.5

with open("config.json", "r") as f:
        headers = load_json(f)
headers

{'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
 'Accept-Language': 'en-US,en;q=0.9',
 'Accept-Encoding': 'gzip, deflate, br',
 'Connection': 'keep-alive',
 'Upgrade-Insecure-Requests': '1',
 'Sec-Fetch-Dest': 'document',
 'Sec-Fetch-Mode': 'navigate',
 'Sec-Fetch-Site': 'none',
 'Sec-Fetch-User': '?1',
 'Cache-Control': 'max-age=0',
 'DNT': '1'}

In [123]:
with open("../../data/raw/urls/NBA_Leagues.txt", "r") as f:
    season_url = f.read().splitlines()
    
with open("../../data/raw/urls/teams_url.txt", "r") as f:
    teams_url = f.read().splitlines()

# Team page
# Conference_body
# per_game_body
# total_body
# advanced_body
                        
columns = [
        "ID", # main
        "Season", # main
        "Team Name",
        
        "Arena",								# advanced_body
        "Games (G)",								# total_body
        "Wins (W)",								# Conference_body 5
        "Losses (L)",								# Conference_body
        "Win/Loss Percentage (W/L%)",				# Conference_body
        "Minutes Played (MP)",						# total_body
        "Pace Factor (Pace)",						# advanced_body
        "Relative Pace",							# Team page 10
        "Offensive Rating (ORtg)",					# advanced_body
        "Relative Offensive Rating",					# Team page
        "Defensive Rating (DRtg)",					# advanced_body
        "Relative Defensive Rating",					# Team page
        "Points Per Game (Pts/G)",				      # Conference_body 15
        "Opponent Points Per Game (Opp Pts/G)",		  # Conference_body
        "Field Goals Made (FG)",					# total_body
        "Field Goal Attempts (FGA)",					# total_body
        "Field Goal Percentage (FG%)",					# total_body
        "3-Point Field Goals Made (3P)",			# total_body 20
        "3-Point Field Goal Attempts (3PA)",			# total_body
        "3-Point Field Goal Percentage (3P%)",			# total_body
        "2-Point Field Goals Made (2P)",			# total_body
        "2-Point Field Goal Attempts (2PA)",			# total_body
        "2-Point Field Goal Percentage (2P%)",			# total_body 25
        "Effective Field Goal Percentage (eFG%)",		# advanced_body
        "Free Throws Made (FT)",					# total_body
        "Free Throw Attempts (FTA)",					# total_body
        "Free Throw Percentage (FT%)",					# total_body
        "FT/FGA",									# advanced_body 30
        "Offensive Rebounds (ORB)",					# total_body
        "Offensive Rebound Percentage (ORB%)",			# advanced_body
        "Defensive Rebounds (DRB)",					# total_body
        "Defensive Rebound Percentage (DRB%)",			# advanced_body
        "Total Rebounds (TRB)",							# total_body 35
        "Total Rebounds Per Game",					# per_game_body
        "Total Rebound Percentage (TRB%)",			# ?
        "Assists (AST)",							# total_body
        "Assist Percentage (AST%)",					# ?
        "Assists Per Game",							# per_game_body 40
        "Steals (STL)",								# total_body
        "Steal Percentage (STL%)",					# ?
        "Steals Per Game",							# per_game_body
        "Blocks (BLK)",								# total_body
        "Block Percentage (BLK%)",					# ? 45
        "Blocks Per Game",							# per_game_body
        "Turnovers (TOV)",							# total_body
        "Turnover Percentage (TOV%)",					# ?
        "Turnovers Per Game",						# per_game_body
        "Personal Fouls (PF)",							# total_body 50
        "Personal Fouls Per Game",					# per_game_body
        "Points (PTS)",								# total_body
        "Points Per Game"							# per_game_body
]

team_page_columns = [
	[columns[1], "season"],			# "Season", # main
	[columns[2], "team_name"],			# "Team Name",			
	[columns[10], "pace_rel"],			# "Relative Pace",							# Team page
	[columns[12], "off_rtg_rel"],			# "Relative Offensive Rating",					# Team page
	[columns[14], "def_rtg_rel"]			# "Relative Defensive Rating",					# Team page
]

conference_columns = [
	[columns[2], "team_name"],	        # "Team Name",
				
	[columns[5], "wins"],		        	# "Wins (W)",								# Conference_body
	[columns[6], "losses"],			        # "Losses (L)",								# Conference_body
	[columns[7], "win_loss_pct"],   	# "Win/Loss Percentage (W/L%)",				# Conference_body
	[columns[15], "pts_per_g"],			# "Points Per Game (Pts/G)",				      # Conference_body
	[columns[16], "opp_pts_per_g"]		# "Opponent Points Per Game (Opp Pts/G)",	  # Conference_body
]

per_game_columns = [
	[columns[2], "team"],			# "Team Name",
				
	[columns[36], "trb"],			# "Total Rebounds Per Game",					# per_game_body
	[columns[40], "ast"],			# "Assists Per Game",							# per_game_body
	[columns[43], "stl"],			# "Steals Per Game",							# per_game_body
	[columns[46], "blk"],			# "Blocks Per Game",							# per_game_body
	[columns[49], "tov"],			# "Turnovers Per Game",						# per_game_body
	[columns[51], "pf"],			# "Personal Fouls Per Game",					# per_game_body
	[columns[53], "pts"]			# "Points Per Game",					# per_game_body
]

total_columns = [
        [columns[2], "team"],			# "Team Name",
        
        [columns[4], "g"],			# "Games (G)",								# total_body
        [columns[8], "mp"],			# "Minutes Played (MP)",					# total_body
        [columns[17], "fg"],			# "Field Goals Made (FG)",					# total_body
        [columns[18], "fga"],			# "Field Goal Attempts (FGA)",					# total_body
        [columns[19], "fg_pct"],		# "Field Goal Percentage (FG%)",				# total_body
        [columns[20], "fg3"],			# "3-Point Field Goals Made (3P)",			# total_body
        [columns[21], "fg3a"],			# "3-Point Field Goal Attempts (3PA)",			# total_body
        [columns[22], "fg3_pct"],		# "3-Point Field Goal Percentage (3P%)",		# total_body
        [columns[23], "fg2"],			# "2-Point Field Goals Made (2P)",			# total_body
        [columns[24], "fg2a"],			# "2-Point Field Goal Attempts (2PA)",			# total_body
        [columns[25], "fg2_pct"],		# "2-Point Field Goal Percentage (2P%)",		# total_body
        [columns[27], "ft"],			# "Free Throws Made (FT)",					# total_body
        [columns[28], "fta"],			# "Free Throw Attempts (FTA)",					# total_body
        [columns[29], "ft_pct"],		# "Free Throw Percentage (FT%)",				# total_body
        [columns[31], "orb"],			# "Offensive Rebounds (ORB)",					# total_body
        [columns[33], "drb"],			# "Defensive Rebounds (DRB)",					# total_body
        [columns[35], "trb"],			# "Total Rebounds (TRB)",						# total_body
        [columns[38], "ast"],			# "Assists (AST)",							# total_body
        [columns[41], "stl"],			# "Steals (STL)",							# total_body
        [columns[44], "blk"],			# "Blocks (BLK)",							# total_body
        [columns[47], "tov"],			# "Turnovers (TOV)",							# total_body
        [columns[50], "pf"],			# "Personal Fouls (PF)",			        		# total_body
        [columns[52], "pts"]		        # "Points (PTS)",							# total_body
]

advanced_columns = [
        [columns[2], "team"],			# "Season", # main
        
        [columns[3], "arena_name"],			# "Arena",								# advanced_body 3
        [columns[9], "pace"],			# "Pace Factor (Pace)",						# advanced_body 9
        [columns[11], "off_rtg"],			# "Offensive Rating (ORtg)",					# advanced_body 11
        [columns[13], "def_rtg"],			# "Defensive Rating (DRtg)",					# advanced_body 13
        [columns[26], "efg_pct"],			# "Effective Field Goal Percentage (eFG%)",	        # advanced_body 26
        [columns[30], "ft_rate"],			# "FT/FGA",								# advanced_body 30
        [columns[32], "orb_pct"],			# "Offensive Rebound Percentage (ORB%)",	# advanced_body 32
        [columns[34], "drb_pct"]			# "Defensive Rebound Percentage (DRB%)",	# advanced_body 34
]

team_season_data = pd.DataFrame(columns = columns)
team_season_data_p1 = pd.DataFrame(columns = [x for x in columns if x not in [y[0] for y in team_page_columns]] + columns[1:3])
team_season_data_p2 = pd.DataFrame(columns = [y[0] for y in team_page_columns])

In [135]:
def get_data(row: Tag, data_stat: str):
        try:
                return row.find(attrs={"data-stat": data_stat}).a.text.strip()
        except:
                try:
                        return row.find(attrs={"data-stat": data_stat}).text.strip()
                except:
                        pass

def get_info(body: Tag, slc: list, row_ignore: Optional[List] = None) -> pd.DataFrame:
        result = pd.DataFrame(columns = [col for [col, _] in slc])
        info = [row for row in body.select("tr") if (row.get("class") == None or "sr_added_headers" not in row.get("class"))]
        
        for row in info:
                if (row_ignore != None and get_data(row = row, data_stat = row_ignore[0]) != row_ignore[1]):
                        continue
                row_info = {}
                for [col, data_stat] in slc:
                        row_info[col] = get_data(row = row, data_stat = data_stat)
                result.loc[len(result)] = row_info
        return result

def get_team_season_info(url: str, headers: dict, columns: list, start_id: int):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        result = pd.DataFrame()
        
        while True:
                try:
                        Eastern_Conference_body = soup.select_one("#confs_standings_E > tbody")
                        Western_Conference_body = soup.select_one("#confs_standings_W > tbody")
                        per_game_body = soup.select_one("#per_game-team > tbody")
                        total_body = soup.select_one("#totals-team > tbody")
                        advanced_body = soup.select_one("#advanced-team > tbody")
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
        
        ec_df = get_info(Eastern_Conference_body, conference_columns)
        wc_df = get_info(Western_Conference_body, conference_columns)
        pg_df = get_info(per_game_body, per_game_columns)
        t_df = get_info(total_body, total_columns)
        a_df = get_info(advanced_body, advanced_columns)
        conf_df = pd.concat([ec_df, wc_df], ignore_index = True)
        result = conf_df.merge(pg_df, on = columns[2])
        result = result.merge(t_df, on = columns[2])
        result = result.merge(a_df, on = columns[2])
        
        result[columns[0]] = result.reset_index().index
        
        time.sleep(_Delay)
        return result

In [136]:
row_id = 0
for i in tqdm(range(len(season_url))):
        result = get_team_season_info(season_url[i], headers, columns, row_id)
        row_id += len(result)
        result[columns[1]] = int(season_url[i][49:53])
        team_season_data_p1 = pd.concat([team_season_data_p1, result], ignore_index = True)
        # team_season_data_p1.to_csv("../../data/raw/team_season_data.csv", index = False)
        
team_season_data_p1.head()

100%|██████████| 6/6 [00:57<00:00,  9.59s/it]


,ID,Arena,Games (G),Wins (W),Losses (L),Win/Loss Percentage (W/L%),Minutes Played (MP),Pace Factor (Pace),Offensive Rating (ORtg),Defensive Rating (DRtg),...,Blocks Per Game,Turnovers (TOV),Turnover Percentage (TOV%),Turnovers Per Game,Personal Fouls (PF),Personal Fouls Per Game,Points (PTS),Points Per Game,Season,Team Name
0,0,Fiserv Forum,73,56,17,.767,17595,105.1,112.4,102.9,...,5.9,1102,NaN,15.1,1431,19.6,8663,118.7,2020,Milwaukee Bucks
1,1,Scotiabank Arena,72,53,19,.736,17380,100.9,111.1,105.0,...,5.0,1067,NaN,14.8,1559,21.7,8118,112.8,2020,Toronto Raptors
2,2,TD Garden,72,48,24,.667,17430,99.5,113.3,107.0,...,5.6,995,NaN,13.8,1553,21.6,8183,113.7,2020,Boston Celtics
3,3,Bankers Life Fieldhouse,73,45,28,.616,17620,98.9,110.0,108.0,...,5.2,967,NaN,13.2,1445,19.8,7989,109.4,2020,Indiana Pacers
4,4,AmericanAirlines Arena,73,44,29,.603,17745,98.3,112.5,109.5,...,4.5,1088,NaN,14.9,1501,20.6,8179,112.0,2020,Miami Heat


In [130]:
def get_team_page_info(url: str, headers: dict, body_selector: str, slc: list):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        result = {}
        
        while True:
                try:
                        body_tag = soup.select_one(body_selector)
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
        # print(body_tag)
        result = get_info(body = body_tag, slc = slc, row_ignore = ["lg_id", "NBA"])
        
        result[columns[1]] = result[columns[1]].apply(lambda s: int(s[0: 2] + s[5: 7]))
        
        time.sleep(_Delay)
        return result

In [131]:
for i in tqdm(range(len(teams_url))):
        # print(teams_url[i])
        result = get_team_page_info(
                url = teams_url[i],
                headers = headers,
                body_selector = "tbody",
                slc = team_page_columns
        )
        
        team_season_data_p2 = pd.concat([team_season_data_p2, result], ignore_index = True)

        # team_season_data_p2.to_csv("../../data/raw/team_season_data.csv", index = False)

100%|██████████| 53/53 [04:15<00:00,  4.81s/it]


In [132]:
team_season_data_p2

,Season,Team Name,Relative Pace,Relative Offensive Rating,Relative Defensive Rating
0,2026,Atlanta Hawks,2.3,0.3,-2.1
1,2025,Atlanta Hawks,3.8,0.1,1.2
2,2024,Atlanta Hawks,1.6,1.9,4.1
3,2023,Atlanta Hawks,1.6,1.8,1.5
4,2022,Atlanta Hawks,-0.5,4.5,2.9
...,...,...,...,...,...
3524,1950,Sheboygan Red Skins,,,
3525,1950,St. Louis Bombers,,,
3526,1951,Washington Capitols,11.6,-1.9,3.0
3527,1950,Washington Capitols,,,


In [ ]:
list(filter(lambda x: x not in team_page_columns, columns))
# team_page_columns

# list2_first = 

[x for x in columns if x not in [y[0] for y in team_page_columns]]


['ID',
 'Arena',
 'Games (G)',
 'Wins (W)',
 'Losses (L)',
 'Win/Loss Percentage (W/L%)',
 'Minutes Played (MP)',
 'Pace Factor (Pace)',
 'Offensive Rating (ORtg)',
 'Defensive Rating (DRtg)',
 'Points Per Game (Pts/G)',
 'Opponent Points Per Game (Opp Pts/G)',
 'Field Goals Made (FG)',
 'Field Goal Attempts (FGA)',
 'Field Goal Percentage (FG%)',
 '3-Point Field Goals Made (3P)',
 '3-Point Field Goal Attempts (3PA)',
 '3-Point Field Goal Percentage (3P%)',
 '2-Point Field Goals Made (2P)',
 '2-Point Field Goal Attempts (2PA)',
 '2-Point Field Goal Percentage (2P%)',
 'Effective Field Goal Percentage (eFG%)',
 'Free Throws Made (FT)',
 'Free Throw Attempts (FTA)',
 'Free Throw Percentage (FT%)',
 'FT/FGA',
 'Offensive Rebounds (ORB)',
 'Offensive Rebound Percentage (ORB%)',
 'Defensive Rebounds (DRB)',
 'Defensive Rebound Percentage (DRB%)',
 'Total Rebounds (TRB)',
 'Total Rebounds Per Game',
 'Total Rebound Percentage (TRB%)',
 'Assists (AST)',
 'Assist Percentage (AST%)',
 'Assists 

In [167]:
team_season_data_p2.drop_duplicates(inplace = True)
result = team_season_data_p1.merge(team_season_data_p2, on = [columns[1], columns[2]])
result = result[columns]

In [ ]:
def full_astype(df: pd.DataFrame):
        # int
        lst = [0, 1, 4, 5, 6, 8, 17, 18, 20, 21, 23, 24, 27, 28, 31, 33, 35, 38, 41, 44, 47, 50, 52]
        for i in lst:
                df[columns[i]] = df[columns[i]].astype("int")
        # float
        lst = [7, 9, 10, 11, 12, 13, 14, 15, 16, 19, 22, 25, 26, 29, 30, 32, 34, 36, 37, 39, 40, 42, 43, 45, 46, 48, 49, 51, 53]
        for i in lst:
                df[columns[i]] = df[columns[i]].astype("float")
        # str
        lst = [2, 3]
        for i in lst:
                df[columns[i]] = df[columns[i]].astype("str")
full_astype(result)
result.to_csv("../../data/raw/team_season_data.csv", index = False)
result.head()

In [165]:
result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 54 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   ID                                      180 non-null    object
 1   Season                                  180 non-null    object
 2   Team Name                               180 non-null    object
 3   Arena                                   180 non-null    object
 4   Games (G)                               180 non-null    object
 5   Wins (W)                                180 non-null    object
 6   Losses (L)                              180 non-null    object
 7   Win/Loss Percentage (W/L%)              180 non-null    object
 8   Minutes Played (MP)                     180 non-null    object
 9   Pace Factor (Pace)                      180 non-null    object
 10  Relative Pace                           180 non-null    object
 11  Offens

In [159]:
team_season_data_p2[(team_season_data_p2["Team Name"] == "Atlanta Hawks") & (team_season_data_p2["Season"] == 2020)]

,Season,Team Name,Relative Pace,Relative Offensive Rating,Relative Defensive Rating
6,2020,Atlanta Hawks,2.7,-3.4,4.2
160,2020,Atlanta Hawks,2.7,-3.4,4.2
1842,2020,Atlanta Hawks,2.7,-3.4,4.2


In [163]:
team_season_data_p2

,Season,Team Name,Relative Pace,Relative Offensive Rating,Relative Defensive Rating
0,2026,Atlanta Hawks,2.3,0.3,-2.1
1,2025,Atlanta Hawks,3.8,0.1,1.2
2,2024,Atlanta Hawks,1.6,1.9,4.1
3,2023,Atlanta Hawks,1.6,1.8,1.5
4,2022,Atlanta Hawks,-0.5,4.5,2.9
...,...,...,...,...,...
3524,1950,Sheboygan Red Skins,,,
3525,1950,St. Louis Bombers,,,
3526,1951,Washington Capitols,11.6,-1.9,3.0
3527,1950,Washington Capitols,,,
